IMPORTS

In [71]:
import pandas as pd
import ollama
import yaml
import litellm

TESTING

In [72]:
test = pd.read_csv("./radiology-reporting-harness/test.csv")
test.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation
0,00320399-b3ed-447c-be07-a3e3b9af6e2e,CT,Abdomen,CT A/P WO-B,60-64,female,FINDINGS:\nLIVER: Normal in size and attenuati...,Surgically absent gallbladder. Surgical clips ...
1,01b1c9ab-800b-4963-b11d-9d503952e57c,XRAY,Lumbar spine,XR LSP 4V,40-44,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,Mild lower lumbar facet degenerative changes.\...
2,02aae38a-088b-4cae-bfdc-f0fb38da565d,XRAY,Thoracic spine,XR TSP 2V,50-54,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,"degen chnges , mild scoliosis convexity to left"
3,045ea333-6d30-4edf-9d33-8ed3feb0772d,MRI,Lumbar spine,MRI LUMBAR,50-54,male,FINDINGS:\nVERTEBRAE: Normal vertebral body he...,Straightening of the normal lumbar lordosis is...
4,06ef946f-5b3b-4b7c-a64f-58c184dca4e6,XRAY,Shoulder,XR L SHOULDER 2V,65-69,male,FINDINGS:\nBONES: No fracture or focal lesion....,Bones show no acute fracture or dislocation. G...


In [73]:
test['template_content'][0].split("\n")

['FINDINGS:',
 'LIVER: Normal in size and attenuation. No focal hepatic lesion.',
 'GALLBLADDER AND BILIARY TREE: Normal. No biliary ductal dilatation.',
 'PANCREAS: Normal in size and contour.',
 'SPLEEN: Normal.',
 'ADRENAL GLANDS: Normal in size and morphology.',
 'KIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.',
 'URINARY BLADDER: Normal. No calculus or mass lesion.',
 'REPRODUCTIVE: The uterus and ovaries/prostate are normal in appearance.',
 'MAJOR VESSELS: The abdominal aorta is normal in caliber.',
 'PERITONEUM: No ascites. No pneumoperitoneum.',
 'ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.',
 'LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.',
 'STOMACH AND BOWEL: No evidence of bowel obstruction.',
 'BONES: No acute osseous abnormality.',
 '',
 'IMPRESSION:',
 'No acute intra-abdominal abnormality identified.']

In [74]:
test['dictation'][0].split("\n")

['Surgically absent gallbladder. Surgical clips are seen in the gallbladder fossa. ',
 'small splenic calcification',
 'pelvic phleboltihs',
 'appendix is surgically absent, with surgical suture at the cecum.',
 'Colonic diverticulosis is present. There is focal moderate pericolonic fat stranding adjacent to the sigmoid colon, best seen on series 201 images 180 to 185, consistent with acute diverticulitis. No adjacent fluid collection or abscess is identified. No extraluminal free air is seen.',
 'no bowel obstruction',
 'Enteric contrast is seen from stomach to distal small']

In [75]:
train = pd.read_csv("./radiology-reporting-harness/train.csv")
train.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation,report
0,2c88d015-b359-4e2c-9c62-61a23ce9d1c3,XRAY,Hip,XR RT HIP 2V,90+,female,FINDINGS:\nBONES: No acute fracture or focal o...,No acute fracture or dislocation. Mild right h...,FINDINGS:\nBONES: No acute fracture. Mild dege...
1,218fc4a5-c7e9-44a7-9aea-54e56bb22663,XRAY,Chest,XR CXR 2V,65-69,male,FINDINGS:\nLUNGS: Lungs are clear. No focal ai...,"no effusuon, infiltrates\nmild thoracic spondy...",FINDINGS:\nLUNGS: Lungs are clear. No focal ai...
2,2066ed76-9f41-4ac8-95f6-74ec1e4cef8a,MRI,Shoulder,MRI RT SHOULDER WO,45-49,male,FINDINGS:\nTENDONS:\nSUPRASPINATUS: The tendon...,MRI RIGHT SHOULDER WITHOUT CONTRAST Right shou...,FINDINGS:\nTENDONS:\nSUPRASPINATUS: There is m...
3,71210e04-aa16-4a47-8c20-fe58494260dc,MRI,Head,MRI Brain^SUBTLE BRAIN,70-74,female,FINDINGS:\nBRAIN: No restricted diffusion to i...,"mild atrophy , mild leuko",FINDINGS:\nBRAIN: There is mild atrophy. There...
4,656574f1-ffea-49f7-a0c8-953a5cd40bfc,MRI,Pelvis,MRI PELVIS,45-49,male,FINDINGS:\nBOWEL: The visualized loops of smal...,The examination is suboptimal due to multiple ...,FINDINGS:\nThe examination is suboptimal due t...


In [76]:
train['template_content'][0].split("\n")

['FINDINGS:',
 'BONES: No acute fracture or focal osseous lesion.',
 'JOINTS: No dislocation. The joint spaces are normal.',
 'SOFT TISSUES: The soft tissues are unremarkable.',
 '',
 'IMPRESSION:',
 'No acute osseous abnormality.']

In [77]:
train['dictation'][0].split("\n")

['No acute fracture or dislocation. Mild right hip osteoarthrosis with mild superolateral joint space narrowing and small marginal acetabular and femoral head osteophytes. Mild degenerative changes of the bilateral sacroiliac joints Mild degenerative changes of the visualized lower lumbar spine. Pelvic ring is intact.']

OLLAMA CLIENT LLM AND EXECUTION

In [95]:
def get_prompt(inputs, version="v1"):
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    prompt_file = config["prompt"][version]["name"]

    with open(f'./prompts/{prompt_file}', "r", encoding="utf-8") as f:
        prompt_text = f.read()

    prompt = prompt_text.format(
        modality=inputs["modality"],
        body_part=inputs["body_part"],
        study_description=inputs["study_description"],
        patient_age_band=inputs["patient_age_band"],
        patient_sex=inputs["patient_sex"],
        template_content=inputs["template_content"],
        dictation=inputs["dictation"],
        few_shot=few_shot
    )
    
    return prompt

def get_model(version="m1"):
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return {"model": config["model"][version]["name"], "base": config["model"][version]["base"]}

# prompt = get_prompt({
#     "modality": test['modality'][0],
#     "body_part": test['body_part'][0],
#     "study_description": test['study_description'][0],
#     "patient_age_band": test['patient_age_band'][0],
#     "patient_sex": test['patient_sex'][0],
#     "template_content": test['template_content'][0],
#     "dictation": test['dictation'][0]
# })

# response = client.generate(
#         model="deepseek-r1:1.5b",
#         prompt=prompt,
#         stream=False
#     )

# response["response"]

In [ ]:
from litellm import completion

few_shot_input = train.iloc[0].drop("report").to_string()
few_shot_output = train.iloc[0]["report"]

few_shot = f"Input:\n {few_shot_input}\nOutput:\n {few_shot_output}"

def callLLM(inputs):

    prompt = get_prompt(inputs)
    model_info = get_model()
    
    for chunk in completion(
    model=model_info["model"],
    base=model_info["base"],
    messages=[{"role": "user", "content": prompt}],
    stream=True,
    ):
        response = chunk.choices[0].delta.content or ""
    return response

# callLLM({
#     "modality": test['modality'][0],
#     "body_part": test['body_part'][0],
#     "study_description": test['study_description'][0],
#     "patient_age_band": test['patient_age_band'][0],
#     "patient_sex": test['patient_sex'][0],
#     "template_content": test['template_content'][0],
#     "dictation": test['dictation'][0],
#     "few_shot": few_shot
# })

    # return response["response"]

FINDINGS:
- Gallbladder absence, surgically clipped.
- Splenic calcification.
- Pelvic phle boltihs.
- Absorbing colon, focal moderate pericolonic fat stranding adjacent to the sigmoid colon.
- Absorbing colon.

IMPRESSION:
1. Gallbladder absence, which has been surgically clipped.
2. Splenic calcification.
3. Pelvic phle boltihs.
4. Absorbing colon, focal moderate pericolonic fat stranding adjacent to the sigmoid colon.
5. Absorbing colon.

In [99]:
ans = []

for i in range(1):
    res = callLLM(test.iloc[i].drop("case_id").to_dict())
    ans.append({"case_id": test.iloc[i].case_id, "report": res})
    print(res)

FINDINGS:
GALLBLADDER AND BILIARY TREE: No biliary ductal dilatation. Mild right hip osteoarthrosis is present with mild superolateral joint space narrowing and small marginal acetabular and femoral head osteophytes. Mild degenerative changes of the bilateral sacroiliac joints are noted. There is no dislocation.

SPLEEN: No calculus or mass lesion. Colonic diverticulosis is present. There is focal moderate pericolonic fat stranding adjacent to the sigmoid colon. No extraluminal free air is seen.

ABODEX Wall: Unremarkable. No hernia.

LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.

STOMACH AND BOWEL: No evidence of bowel obstruction.

IMPRESSION:
1. Mild osteoarthrosis of the right hip.
2. Mild degenerative changes of sacroiliac joints.
3. No dislocation.None


In [100]:
Here is the edited radiology report:

<OUTPUT>
FINDINGS:
GALLBLADDER AND BILIARY TREE: Surgically absent gallbladder. Surgical clips are seen in the gallbladder fossa.
SPLEEN: Small splenic calcification.
KIDNEYS AND URETERS: No significant findings.
URINARY BLADDER: No calculus or mass lesion.
REPRODUCTIVE: The uterus and ovaries/prostate are normal in appearance.
MAJOR VESSELS: The abdominal aorta is normal in caliber.
PERITONEUM: No ascites. No pneumoperitoneum.
ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.
LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.
STOMACH AND BOWEL: Colonic diverticulosis is present. There is focal moderate pericolonic fat stranding adjacent to the sigmoid colon, best seen on series 201 images 180 to 185, consistent with acute diverticulitis. No adjacent fluid collection or abscess is identified. No extraluminal free air is seen.
BONES: No acute osseous abnormality.
OTHER FINDINGS: None.

IMPRESSION:
The findings are consistent with surgically absent gallbladder, small splenic calcification, colonic diverticulosis with acute diverticulitis, and enteric contrast seen from stomach to distal small bowel. No acute intra-abdominal abnormality identified.
</OUTPUT>
Based on the input, I will convert the dictation into a completed structured radiology report by modifying the supplied normal template.

**Output:**

FINDINGS:
- VERTAEBAE: Normal density and alignment. However, there are mild lower lumbar facet degenerative changes.
- DISC SPACES: Preserved.
- SOFT TISSUES: Unremarkable.
- OTHER FINDINGS: None.

IMPRESSION:
Mild lower lumbar facet degenerative changes.

I followed the rules as specified:

1. Started with the template as the base report.
2. Routed the dictated finding to the corresponding FINDINGS field.
3. Modified the normal statement for VERTAEBAE to reflect the abnormality.
4. Preserved template statements for routinely visualized regions (DISC SPACES and SOFT TISSUES) that were not mentioned in the dictation.
5. Updated the IMPRESSION to summarize only the important abnormal finding.
6. Did not add findings unsupported by the dictation or template.
7. Preserved the template wording and field order whenever possible.
8. Structured the output exactly as FINDINGS and IMPRESSION.
9. Used the OTHER FINDINGS field only for relevant findings that do not belong elsewhere (in this case, none).
10. Used the context information only for context and not for additional findings.
Based on the input, I will convert the dictation into a completed structured radiology report.

**OUTPUT**

FINDINGS:
VERTTEBRAE: 
  Density: Not mentioned in the dictation, so no change.
  Alignment: Mild scoliosis convexity to left.
  Fracture or osseous lesion: No change.
  
DISC SPACES: Preserved.
  
SOFT TISSUES: Unremarkable.

IMPRESSION:
Mild scoliosis convexity to the left.

Explanation:

* Since the dictation only mentions "degen chnges" (degenerative changes) and "mild scoliosis convexity to left", I have only modified the corresponding fields in the FINDINGS section.
* I preserved the normal statement for density and the absence of fracture or osseous lesion.
* The preserved normal statements for disc spaces and soft tissues were not mentioned in the dictation, so they remain unchanged.
* The IMPRESSION only summarizes the important abnormal finding, which is the mild scoliosis convexity to the left.
FINDINGS:
VERTEBRAE: No acute fracture or focal osseous lesion, but a benign-appearing vertebral hemangioma is noted within the L2 vertebral body.
ALIGNMENT: The normal anatomic lumbar lordosis is maintained, but it is straightened.
SPINAL CORD: The conus medullaris terminates at a normal level, and the cauda equina nerve roots are unremarkable, with no significant focal compression.
DISCS/DEGENERATIVE CHANGES: Multilevel degenerative spondylotic changes are seen in the form of marginal osteophyte formation and multilevel intervertebral disc desiccation/bulging. At L1-L2, a mild diffuse disc bulge is noted without significant central canal stenosis or neural foraminal compromise. At L2-L3, diffuse disc bulge is noted with bilateral neural foraminal narrowing. At L3-L4, diffuse disc bulge with bilateral foraminal extension is noted, causing bilateral neural foraminal narrowing with crowding/impingement of the exiting nerve roots. At L4-L5, diffuse disc bulge with bilateral neural foraminal narrowing is noted. At L5-S1, diffuse disc bulge with bilateral foraminal extension is noted, causing bilateral neural foraminal narrowing and impingement of the exiting nerve roots.
L1-L2: The intervertebral disc is normal in appearance, except for mild diffuse disc bulge.
L2-L3: The intervertebral disc is normal in appearance, except for diffuse disc bulge with bilateral neural foraminal narrowing.
L3-L4: The intervertebral disc is normal in appearance, except for diffuse disc bulge with bilateral foraminal extension and bilateral neural foraminal narrowing with crowding/impingement of the exiting nerve roots.
L4-L5: The intervertebral disc is normal in appearance, except for diffuse disc bulge with bilateral neural foraminal narrowing.
L5-S1: The intervertebral disc is normal in appearance, except for diffuse disc bulge with bilateral foraminal extension and bilateral neural foraminal narrowing with impingement of the exiting nerve roots.
PARASPINAL SOFT TISSUES: The visualized paravertebral soft tissues are symmetric and without abnormal signal, except for focal marrow/soft-tissue edema-like signal involving the ANTERIOR ADJACENT ENDPLATE at L2-L3 AND SPINOUS PROCESS OF L4.

IMPRESSION:
Multilevel lumbar spondylotic degenerative changes with multilevel disc bulges, L3-L4 bilateral neural foraminal narrowing with impingement of bilateral exiting nerve roots, L4-L5 disc bulge with mild central canal stenosis (canal AP diameter ~9.6 mm) and impingement of bilateral exiting nerve roots, L5-S1 bilateral foraminal narrowing with impingement of bilateral exiting nerve roots, and a vertebral hemangioma in L2.
Here is the completed structured radiology report:


<OUTPUT>
FINDINGS:
BONES: Mild diffuse osseous demineralization.
JOINTS: No dislocation. The joint spaces and articular margins are normal.
SOFT TISSUES: The soft tissues are unremarkable.
OTHER FINDINGS: None

IMPRESSION:
Mild diffuse osseous demineralization and mild acromioclavicular osteoarthritis.
</OUTPUT>
Here is the modified radiology report based on the provided template and dictation:


<OUTPUT>
FINDINGS:
VERTEBRAE: No acute fracture, subluxation, or aggressive osseous lesion. Vertebral body heights are maintained.
ALIGNMENT: The thoracic kyphotic curvature is exaggerated.
DISCS/DEGENERATIVE CHANGES: Multilevel degenerative spondylotic changes with disc desiccation and small marginal osteophyte formation. No significant disc bulge, disc extrusion, or significant posterior disc protrusion is identified.
SPINAL CORD: The thoracic spinal cord is normal in caliber and morphology, with no focal cord indentation or compression, and no abnormal intramedullary T2/STIR hyperintense signal or cord edema.
PARASPINAL SOFT TISSUES: The paravertebral soft tissues demonstrate normal signal and morphology.

IMPRESSION:
The MRI of the thoracic spine shows exaggerated thoracic kyphotic curvature and multilevel mild thoracic spondylotic degenerative changes with small marginal osteophytes, with no significant disc protrusion, spinal canal stenosis, or neural foraminal compromise.
</OUTPUT>
Here is the final structured radiology report:


FINDINGS:
BONES: No acute fracture or focal osseous lesion. Possible fracture of the left lesser trochanter and possible fracture of the left femur.
JOINTS: Bilateral hip joints appear abnormal with possible dislocation. The joint spaces are abnormal.
SOFT TISSUES: The soft tissues are unremarkable.
OTHER FINDINGS: None

IMPRESSION:
Possible fractures of the left lesser trochanter and left femur, and possible bilateral hip joint dislocation.
FINDINGS:
BONES: No acute fracture or focal osseous lesion.
JOINTS: Mild superolateral joint space narrowing, subchondral/endplate sclerosis, and marginal osteophyte formation are present in the left hip joint. Degenerative arthropathy with subchondral sclerosis and mild joint space irregularity are present in the left sacroiliac joint. 
SOFT TISSUES: The soft tissues are unremarkable.

IMPRESSION:
Mild left hip osteoarthritic changes and left sacroiliac joint arthropathy.
FINDINGS:
VERTEBRAE: Abnormal marrow signal intensity involving the T7, T8, and T9 vertebral bodies, with predominant involvement of the T8 vertebral body, appearing hyperintense on T1- and T2-weighted images and showing hyperintensity on STIR sequences. The appearance is suggestive of vertebral hemangiomatous changes, particularly at T7 and T8, with predominant involvement of T8. 
Mild STIR hyperintense signal is also seen along the inferior endplate of T8 and superior endplate of T9, with mild adjacent endplate irregularity. These changes are likely reactive/degenerative in nature and may represent stress-related or repetitive traumatic changes.
DEGENERATIVE CHANGES: Multilevel mild degenerative spondylotic changes are seen in the dorsal spine with marginal osteophyte formation and associated disc desiccation. No significant focal disc protrusion or extrusion is seen in the dorsal spine.
DISCS/DEGENERATIVE CHANGES: The intervertebral disc heights and signal are preserved in the thoracic spine. No significant disc bulge or herniation is seen.
ALIGNMENT: The anatomic alignment of the thoracic spine is preserved.
SPINAL CORD: The thoracic spinal cord is normal in signal intensity and caliber. No significant cervical spinal cord indentation is evident on the provided images.
PARASPINAL SOFT TISSUES: The paravertebral soft tissues demonstrate normal signal and morphology.
OTHER FINDINGS:


IMPRESSION:
The MRI of the thoracic spine demonstrates vertebral hemangiomatous changes, particularly at T7 and T8, with predominant involvement of T8. Mild degenerative changes are also seen in the dorsal spine. Contrast-enhanced MRI of the dorsal spine is recommended for further characterization of the atypical/heterogeneous marrow signal and to confirm benign hemangiomatous etiology.
FINDINGS:
LOWER THORAX: No pleural effusion or basilar consolidation.
LIVER: Mildly enlarged, measuring approximately 16 cm in craniocaudal span. No focal hepatic lesion identified on the provided sequences.
GALLBLADDER AND BILE DUCTS: The gallbladder is not visualized, consistent with postoperative status (cholecystectomy). The common bile duct is normal in caliber with no evidence of intraluminal filling defect or obstructive dilatation. No intrahepatic biliary ductal dilatation.
PANCREAS: Normal in size and signal intensity. The main pancreatic duct is not dilated. No focal pancreatic lesion is seen.
SPLEEN: Mildly enlarged, measuring approximately 12.2 cm. No significant free fluid or obvious upper abdominal lymphadenopathy is seen.
ADRENALS: The adrenal glands are normal in size and configuration.
KIDNEYS: Normal in size, position, and signal intensity. No hydronephrosis or suspicious renal mass.
STOMACH AND BOWEL: The visualized portions of the stomach and bowel are unremarkable. No bowel wall thickening or obstruction.
LYMPH NODES: No retroperitoneal or mesenteric lymphadenopathy.
VASCULATURE: The abdominal aorta and major visceral vessels are patent. No evidence of aneurysm or dissection.
OTHER FINDINGS:


IMPRESSION:
The patient has mild hepatomegaly and splenomegaly consistent with post-cholecystectomy status. No biliary dilatation or choledocholithiasis is seen. The pancreas and main pancreatic duct are unremarkable.

SyntaxError: invalid syntax (1746401558.py, line 1)

In [ ]:
submission = pd.DataFrame(ans)
submission.to_csv("submission.csv", index=False)